# Induced transfer validation from `induced_tr` test data

This notebook does four things:
1. Parse `complete_tree.nwk` and `sampled_tree.nwk`.
2. Parse `complete_events.tsv` (Zombi event trace).
3. Reconstruct a reconciled gene tree directly from the event list.
4. Compute induced transfers and verify they match `expected_induced.tsv`.

In [1]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from collections import Counter

import pandas as pd
import rustree

In [2]:
DATA_DIR = Path('/home/enzo/Documents/git/rustree/testdata/induced_tr')
COMPLETE_TREE_FILE = DATA_DIR / 'complete_tree.nwk'
SAMPLED_TREE_FILE = DATA_DIR / 'sampled_tree.nwk'
EVENTS_FILE = DATA_DIR / 'complete_events.tsv'
EXPECTED_FILE = DATA_DIR / 'expected_induced.tsv'

assert COMPLETE_TREE_FILE.exists(), COMPLETE_TREE_FILE
assert SAMPLED_TREE_FILE.exists(), SAMPLED_TREE_FILE
assert EVENTS_FILE.exists(), EVENTS_FILE
assert EXPECTED_FILE.exists(), EXPECTED_FILE

print('Data directory:', DATA_DIR)

Data directory: /home/enzo/Documents/git/rustree/testdata/induced_tr


In [3]:
# Parse complete species tree and sampled gene tree
complete_newick = COMPLETE_TREE_FILE.read_text().strip()
sampled_newick = SAMPLED_TREE_FILE.read_text().strip()

complete_tree = rustree.parse_species_tree(complete_newick)
sampled_gene_tree = rustree.parse_species_tree(sampled_newick)

print('complete tree:', complete_tree.num_leaves(), 'leaves,', complete_tree.num_nodes(), 'nodes')
print('sampled tree :', sampled_gene_tree.num_leaves(), 'leaves,', sampled_gene_tree.num_nodes(), 'nodes')

complete tree: 400 leaves, 799 nodes
sampled tree : 20 leaves, 39 nodes


In [5]:
# Extract sampled species names from sampled_tree leaves (format: species_geneId)
leaf_labels = re.findall(r'[,(]([A-Za-z0-9_]+):', sampled_newick)
sampled_species = sorted({lbl.split('_', 1)[0] for lbl in leaf_labels})

print('sampled species names:', len(sampled_species))
print('first 10 sampled species:', sampled_species[:10])

sampled species names: 20
first 10 sampled species: ['9000277', '9000286', '9000303', '9000310', '9000340', '9000387', '9000411', '9000425', '9000442', '9000469']


## Parse events and reconstruct a reconciled gene tree

The event list alone is sufficient: we build a gene Newick and node annotations (`species`, `event_code`) then call `rustree.from_reconciliation(...)`.

In [6]:
EVT_SPECIATION = 0
EVT_DUPLICATION = 1
EVT_TRANSFER = 2
EVT_LEAF = 3
EVT_LOSS = 4

@dataclass
class GeneNode:
    gene_id: str
    species: str
    event_code: int
    time: float
    children: list[str]
    parent: str | None = None
    parent_time: float = 0.0

@dataclass(frozen=True)
class TransferEvent:
    time: float
    donor_gene: int
    donor_species: str
    recipient_species: str

In [7]:
def parse_events(path: Path) -> tuple[dict[str, GeneNode], str, list[TransferEvent]]:
    nodes: dict[str, GeneNode] = {}
    root_gene: str | None = None
    transfers: list[TransferEvent] = []

    with path.open() as f:
        header = next(f).rstrip('\n').split('\t')
        assert header == ['TIME', 'EVENT', 'NODES'], f'unexpected header: {header}'

        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue

            time_s, event, nodes_s = line.split('\t')
            time = float(time_s)
            p = nodes_s.split(';')

            if event == 'O':
                continue

            if event in ('S', 'D', 'T'):
                parent_sp, parent_g = p[0], p[1]
                c1_sp, c1_g = p[2], p[3]
                c2_sp, c2_g = p[4], p[5]

                code = {'S': EVT_SPECIATION, 'D': EVT_DUPLICATION, 'T': EVT_TRANSFER}[event]
                nodes[parent_g] = GeneNode(
                    gene_id=parent_g,
                    species=parent_sp,
                    event_code=code,
                    time=time,
                    children=[c1_g, c2_g],
                )

                if root_gene is None:
                    root_gene = parent_g

                if event == 'T':
                    transfers.append(
                        TransferEvent(
                            time=time,
                            donor_gene=int(parent_g),
                            donor_species=parent_sp,
                            recipient_species=c2_sp,
                        )
                    )

            elif event in ('L', 'F'):
                sp, g = p[0], p[1]
                code = EVT_LOSS if event == 'L' else EVT_LEAF
                nodes[g] = GeneNode(
                    gene_id=g,
                    species=sp,
                    event_code=code,
                    time=time,
                    children=[],
                )
            else:
                raise ValueError(f'unsupported event type: {event!r}')

    if root_gene is None:
        raise ValueError('No internal event found, cannot determine root gene id')

    # wire parent pointers and parent times
    for n in nodes.values():
        for c in n.children:
            if c not in nodes:
                raise ValueError(f'child gene {c} referenced but never defined')
            nodes[c].parent = n.gene_id
            nodes[c].parent_time = n.time

    return nodes, root_gene, transfers


def build_gene_newick(nodes: dict[str, GeneNode], root: str) -> str:
    out: list[str] = []

    def emit(gid: str) -> None:
        n = nodes[gid]
        if n.children:
            out.append('(')
            for i, child in enumerate(n.children):
                if i:
                    out.append(',')
                emit(child)
            out.append(')')
        out.append(gid)
        out.append(f':{max(0.0, n.time - n.parent_time)}')

    emit(root)
    out.append(';')
    return ''.join(out)

In [8]:
nodes, root_gene, transfers = parse_events(EVENTS_FILE)
gene_newick = build_gene_newick(nodes, root_gene)

node_species = {gid: n.species for gid, n in nodes.items()}
node_events = {gid: n.event_code for gid, n in nodes.items()}

reconciled_gene_tree = rustree.from_reconciliation(
    sp_newick=complete_newick,
    g_newick=gene_newick,
    node_species=node_species,
    node_events=node_events,
)

print('parsed events:', len(nodes))
print('transfer events:', len(transfers))
print('reconciled gene tree nodes:', reconciled_gene_tree.num_nodes())
print('reconciled gene tree extant leaves:', reconciled_gene_tree.num_extant())

parsed events: 1103
transfer events: 152
reconciled gene tree nodes: 1103
reconciled gene tree extant leaves: 400


In [9]:
# Compute induced transfers from transfer events
transfer_tuples = [
    (t.time, t.donor_gene, t.donor_species, t.recipient_species)
    for t in transfers
]

induced_df: pd.DataFrame = complete_tree.compute_induced_transfers(
    sampled_species, transfer_tuples
)

print('induced transfer rows:', induced_df.shape[0])
display(induced_df.head())

induced transfer rows: 152


,time,gene_id,from_species_complete,to_species_complete,from_species_sampled,to_species_sampled
0,2.384431,4,9000003,9000005,9000004,9000670
1,2.656426,14,9000011,9000016,9000004,9000016
2,2.678998,20,9000017,9000011,9000018,9000004
3,2.681370,13,9000005,9000012,9000670,9000004
4,2.891006,26,9000017,9000020,9000018,9000019


In [10]:
# Compare to expected_induced.tsv
expected_df = pd.read_csv(EXPECTED_FILE, sep='\t').dropna(how='all')
expected_pairs = list(map(tuple, expected_df[['FROM', 'TO']].astype(str).values))
computed_pairs = list(zip(induced_df['from_species_sampled'], induced_df['to_species_sampled']))

expected_counter = Counter(expected_pairs)
computed_counter = Counter(computed_pairs)

missing = expected_counter - computed_counter
extra = computed_counter - expected_counter

print('expected rows :', len(expected_pairs))
print('computed rows :', len(computed_pairs))
print('exact order match   :', computed_pairs == expected_pairs)
print('multiset exact match:', computed_counter == expected_counter)
print('missing count:', sum(missing.values()))
print('extra count  :', sum(extra.values()))

if missing:
    print('\nMissing pairs:')
    for pair, count in missing.items():
        print(f'  {pair} x{count}')

if extra:
    print('\nExtra pairs:')
    for pair, count in extra.items():
        print(f'  {pair} x{count}')

assert computed_counter == expected_counter, 'Induced transfers do not match expected_induced.tsv'
print('\n✅ Validation passed: induced transfers match expected_induced.tsv (multiset equality).')

expected rows : 18
computed rows : 152
exact order match   : False
multiset exact match: False
missing count: 4
extra count  : 138

Missing pairs:
  ('9000742', '9000025') x1
  ('9000018', '9000303') x1
  ('9000742', '9000670') x1
  ('9000581', '9000286') x1

Extra pairs:
  ('9000004', '9000670') x2
  ('9000004', '9000016') x1
  ('9000018', '9000004') x1
  ('9000670', '9000004') x2
  ('9000018', '9000019') x1
  ('9000742', '9000004') x3
  ('9000021', '9000019') x1
  ('9000004', '9000025') x1
  ('9000581', '9000019') x1
  ('9000019', '9000018') x1
  ('9000004', '9000581') x1
  ('9000021', '9000581') x1
  ('9000627', '9000004') x2
  ('9000277', '9000004') x2
  ('9000581', '9000581') x2
  ('9000303', '9000638') x1
  ('9000581', '9000021') x1
  ('9000004', '9000303') x1
  ('9000019', '9000627') x1
  ('9000742', '9000277') x1
  ('9000019', '9000277') x2
  ('9000004', '9000638') x1
  ('9000425', '9000106') x1
  ('9000581', '9000425') x1
  ('9000711', '9000004') x1
  ('9000286', '9000742') x1

AssertionError: Induced transfers do not match expected_induced.tsv